# Volume 11, Chapter 7: Markov and Lévy processes (Python companion)

Exercises first, solutions at the end. Poisson random measures sampled and tested, the ruin of an insurance company, the $\frac12$-stable subordinator built jump by jump from the Lévy–Itô decomposition, Brownian motion run on a random clock, the Gamma subordinator, and a computer science application: **sampling stable laws**, the Chambers–Mallows–Stuck algorithm, which produces an $\alpha$-stable variable from one uniform and one exponential variable, against the Lévy–Itô construction. Requires `numpy`.

In [1]:
import math
import numpy as np
rng = np.random.default_rng(1107)

def prm_uniform(rate, m):
    # m independent Poisson random measures on [0,1]^2 with intensity rate * Lebesgue
    # (Theorem 7.8: a Poisson number of independent uniform points)
    K = rng.poisson(rate, m)
    return [rng.random((k, 2)) for k in K]

## Exercises

**P7.1 (★) Testing a Poisson random measure.** Sample $20000$ Poisson random measures on $[0,1]^2$ with intensity $50\times$Lebesgue. For $A=[0,0.3]^2$ and $A'=[0.5,1]\times[0,1]$, check that $N(A)$ has mean and variance $4.5$, that $N(A')$ has mean and variance $25$, and that $N(A)$ and $N(A')$ are uncorrelated. Then check the exponential formula (Proposition 7.9) for $f(x,y)=\frac{x+y}{10}$.

**P7.2 (★) Ruin.** Claims arrive at rate $\lambda=1$ with exponential sizes of mean $m=1$; premiums come in at rate $c=1.2$. Estimate the probability that the capital $u+ct-X_t$ ever becomes negative, for $u=0,2,5,10$ (use a horizon long enough), and compare with the classical formula $\psi(u)=\frac{\lambda m}c\exp\bigl(-(\frac1m-\frac\lambda c)u\bigr)$ for exponential claims (Exercise 7.7).

**P7.3 (★★) The $\frac12$-stable subordinator, jump by jump.** Build $T_1$ from the Lévy–Itô decomposition (Theorems 7.13 and 7.15): the jumps larger than $\varepsilon$ form a Poisson random measure with $\Pi(]\varepsilon,\infty[)=\sqrt{2/(\pi\varepsilon)}$ points on $[0,1]$, and the jumps smaller than $\varepsilon$ contribute about their mean $\int_0^\varepsilon x\,\Pi(\mathrm dx)=\sqrt{2\varepsilon/\pi}$. Compare the Laplace transform with $e^{-\sqrt{2\lambda}}$ and the quantiles with those of $1/Z^2$, the law of the hitting time of $1$.

**P7.4 (★★) Brownian motion on a random clock.** With $T_a=a^2/Z^2$ independent of a Brownian motion $W$, sample $W_{T_a}$ for $a=1$ and compare its quantiles with those of the Cauchy law (Exercise 7.6). Then sample $W$ at an independent Gamma$(t,1)$ time (the *variance gamma* law of finance): what are its variance and its kurtosis?

**P7.5 (★★) The Gamma subordinator.** With $\Pi(\mathrm dx)=x^{-1}e^{-x}\mathrm dx$, build $X_1$ from its jumps larger than $\varepsilon$ (how many are there on average?) plus the mean of the small ones, for $\varepsilon=10^{-1},10^{-3},10^{-6}$, and compare the mean, variance and a few quantiles with the Gamma$(1,1)$ (exponential) law (Exercise 7.3).

**P7.6 (★★★) CS application: sampling stable laws.** The Chambers–Mallows–Stuck algorithm (1976) returns, for $V$ uniform on $]-\frac\pi2,\frac\pi2[$ and $W$ exponential, the variable
$$X=\frac{\sin(\alpha V)}{(\cos V)^{1/\alpha}}\Bigl(\frac{\cos((1-\alpha)V)}{W}\Bigr)^{(1-\alpha)/\alpha},$$
which is symmetric $\alpha$-stable with $\mathbb Ee^{i\xi X}=e^{-|\xi|^\alpha}$. Check the characteristic function for $\alpha=0.5,1.5,1.9$ (for $\alpha=1$ the formula is $\tan V$, the Cauchy law). Check stability: $(X_1+X_2)/2^{1/\alpha}$ has the law of $X_1$. Finally sample the same law by the Lévy–Itô route, with $\Pi(\mathrm dx)=k_\alpha^{-1}|x|^{-1-\alpha}\mathrm dx$, where $k_\alpha=\frac{2\Gamma(1-\alpha)\cos(\pi\alpha/2)}\alpha$ (Exercise 7.5), and compare the costs.

## Solutions

### P7.1

**How to find it.** Count the points in each set. For the exponential formula, $\mathbb Ee^{-N(f)}=\exp(-50\int(1-e^{-(x+y)/10})\,\mathrm dx\,\mathrm dy)$, and $\int_0^1\int_0^1e^{-(x+y)/10}=100(1-e^{-1/10})^2$.

In [2]:
pts = prm_uniform(50, 20_000)
nA = np.array([np.sum((p[:, 0] < 0.3) & (p[:, 1] < 0.3)) for p in pts])
nB = np.array([np.sum(p[:, 0] >= 0.5) for p in pts])
print(f"N(A):  mean {nA.mean():.3f}, variance {nA.var():.3f}  (4.5)")
print(f"N(A'): mean {nB.mean():.3f}, variance {nB.var():.3f}  (25)")
print(f"correlation {np.corrcoef(nA, nB)[0, 1]:+.4f}")
lhs = np.mean([math.exp(-p.sum() / 10) for p in pts])
rhs = math.exp(-50 * (1 - 100 * (1 - math.exp(-0.1))**2))
print(f"E exp(-N(f)) = {lhs:.4f}, exponential formula {rhs:.4f}")

N(A):  mean 4.515, variance 4.415  (4.5)
N(A'): mean 25.043, variance 24.799  (25)
correlation -0.0012
E exp(-N(f)) = 0.0088, exponential formula 0.0089


Means and variances agree, as they must for Poisson variables, the correlation is zero up to noise ($\pm0.007$), and the exponential formula holds to the precision of the simulation. (With $f=x+y$ instead, $\mathbb Ee^{-N(f)}$ is of order $10^{-13}$ and is carried by rare samples with few points: a Monte Carlo check would need astronomically many samples.)

### P7.2

**How to find it.** Ruin can only happen at a claim time, so simulate the claim times and sizes and test the capital just after each claim. The horizon must be long compared with the time scale of ruin; $500$ time units suffice here, since the capital drifts upwards at rate $c-\lambda m=0.2$.

In [3]:
lam, mclaim, c, horizon, m = 1.0, 1.0, 1.2, 500.0, 4000
n_claims = int(lam * horizon * 1.2) + 50
for u in (0, 2, 5, 10):
    gaps = rng.exponential(1 / lam, (m, n_claims)); times = np.cumsum(gaps, axis=1)
    claims = np.cumsum(rng.exponential(mclaim, (m, n_claims)), axis=1)
    capital = u + c * times - claims
    ruined = np.any((capital < 0) & (times <= horizon), axis=1)
    psi = lam * mclaim / c * math.exp(-(1 / mclaim - lam / c) * u)
    print(f"u = {u:>2}: simulated {ruined.mean():.3f} +- {math.sqrt(ruined.mean() * (1 - ruined.mean()) / m):.3f}   formula {psi:.3f}")

u =  0: simulated 0.831 +- 0.006   formula 0.833
u =  2: simulated 0.605 +- 0.008   formula 0.597


u =  5: simulated 0.369 +- 0.008   formula 0.362
u = 10: simulated 0.161 +- 0.006   formula 0.157


The simulation agrees with the formula, which goes back to Lundberg (1903) and Cramér (1930). Even with a premium $20\%$ above the mean claim rate and no initial capital, ruin happens five times out of six; ten units of capital bring it down to about $16\%$. The exponential decay in $u$ is the Cramér–Lundberg estimate; for heavy-tailed claims (subexponential laws) it fails, and ruin is then typically caused by one huge claim, as in Exercise 7.8(c).

### P7.3

**How to find it.** On $[0,1]$, the number of jumps larger than $\varepsilon$ is Poisson with mean $\Pi(]\varepsilon,\infty[)=\sqrt{2/(\pi\varepsilon)}$, and each has the law $\Pi$ restricted to $]\varepsilon,\infty[$ and normalised, with tail $P(J>x)=\sqrt{\varepsilon/x}$: so $J=\varepsilon/U^2$ with $U$ uniform. Add the mean of the small jumps.

In [4]:
def half_stable_T1(eps, m):
    K = rng.poisson(math.sqrt(2 / (math.pi * eps)), m)
    total = np.array([np.sum(eps / rng.random(k)**2) for k in K])
    return total + math.sqrt(2 * eps / math.pi)

Zq = 1 / rng.standard_normal(10**6)**2
for eps in (1e-1, 1e-3, 1e-5):
    T = half_stable_T1(eps, 40_000)
    lap = [np.mean(np.exp(-lam * T)) for lam in (0.5, 2, 8)]
    print(f"eps = {eps:<6}: Laplace at 0.5, 2, 8: " + ", ".join(f"{v:.4f}" for v in lap)
          + f"   median {np.median(T):.3f}, 90% quantile {np.quantile(T, 0.9):.2f}")
print("exact             : " + ", ".join(f"{math.exp(-math.sqrt(2 * lam)):.4f}" for lam in (0.5, 2, 8))
      + f"   median {np.median(Zq):.3f}, 90% quantile {np.quantile(Zq, 0.9):.2f}")

eps = 0.1   : Laplace at 0.5, 2, 8: 0.3673, 0.1334, 0.0146   median 2.217, 90% quantile 64.01
eps = 0.001 : Laplace at 0.5, 2, 8: 0.3684, 0.1356, 0.0184   median 2.185, 90% quantile 61.24


eps = 1e-05 : Laplace at 0.5, 2, 8: 0.3665, 0.1343, 0.0180   median 2.221, 90% quantile 61.53
exact             : 0.3679, 0.1353, 0.0183   median 2.189, 90% quantile 62.85


With $\varepsilon=10^{-1}$ only about $2.5$ jumps are sampled and the small ones are replaced by their mean. The median, the large quantiles (which come from the large jumps) and the Laplace transform at $\lambda=\frac12$ and $2$ are already right; at $\lambda=8$, which weighs the small values of $T_1$ and therefore the small jumps, the error is about $20\%$, and it disappears at $\varepsilon=10^{-3}$. As $\varepsilon$ decreases, the number of sampled jumps grows like $\varepsilon^{-1/2}$ and the law converges to that of $1/Z^2$: the decomposition of Theorem 7.13 is an algorithm.

### P7.4

**How to find it.** Given $T_a$, $W_{T_a}$ is $N(0,T_a)$: sample $\sqrt{T_a}\,Z'$. For the Gamma clock, $\operatorname{Var}W_G=\mathbb EG=t$ and $\mathbb EW_G^4=3\mathbb EG^2=3(t+t^2)$.

In [5]:
n = 10**6
C = np.sqrt(1 / rng.standard_normal(n)**2) * rng.standard_normal(n)
for q in (0.75, 0.9, 0.99):
    print(f"q = {q}: W at the 1/2-stable clock {np.quantile(C, q):7.3f}   Cauchy {math.tan(math.pi * (q - 0.5)):7.3f}")
for t in (0.5, 2.0):
    G = rng.gamma(t, 1.0, n); V = np.sqrt(G) * rng.standard_normal(n)
    print(f"Gamma clock t = {t}: variance {V.var():.3f} (t = {t}), kurtosis {np.mean(V**4) / V.var()**2:.3f} (3 + 3/t = {3 + 3 / t:.3f})")

q = 0.75: W at the 1/2-stable clock   0.999   Cauchy   1.000
q = 0.9: W at the 1/2-stable clock   3.077   Cauchy   3.078
q = 0.99: W at the 1/2-stable clock  31.788   Cauchy  31.821


Gamma clock t = 0.5: variance 0.500 (t = 0.5), kurtosis 8.995 (3 + 3/t = 9.000)


Gamma clock t = 2.0: variance 1.998 (t = 2.0), kurtosis 4.525 (3 + 3/t = 4.500)


The quantiles are those of the Cauchy law: subordinating Brownian motion by the $\frac12$-stable clock gives the $1$-stable process. With a Gamma clock the result has finite variance $t$ but kurtosis $3(1+\frac1t)$, larger than the Gaussian $3$: Madan and Seneta (1990) proposed this *variance gamma* process for stock returns, whose tails are heavier than Gaussian at short time scales.

### P7.5

**How to find it.** $\Pi(]\varepsilon,\infty[)=E_1(\varepsilon)=\int_\varepsilon^\infty x^{-1}e^{-x}\,\mathrm dx$, computed numerically. Sample jumps from the normalised restriction by inverting its distribution function, tabulated on a grid in $\log x$ (rejection from an exponential proposal would accept with probability about $\varepsilon\log\frac1\varepsilon$, hopeless for small $\varepsilon$). The small jumps contribute $\int_0^\varepsilon e^{-x}\,\mathrm dx=1-e^{-\varepsilon}$ on average.

In [6]:
def jump_table(eps):
    # with x = eps * e^u, Pi(dx) = e^{-x} du: tabulate the cumulative mass in u (trapezoid rule)
    u = np.linspace(0, 6 + math.log(1 / eps), 200_001); x = eps * np.exp(u)
    f = np.exp(-x); cum = np.concatenate([[0], np.cumsum((f[1:] + f[:-1]) / 2 * np.diff(u))])
    return x, cum

for eps in (1e-1, 1e-3, 1e-6):
    x, cum = jump_table(eps); rate = cum[-1]
    K = rng.poisson(rate, 20_000)
    J = np.interp(rng.random(K.sum()) * rate, cum, x)        # all jumps, inverse transform
    X = np.bincount(np.repeat(np.arange(len(K)), K), weights=J, minlength=len(K)) + (1 - math.exp(-eps))
    print(f"eps = {eps:<6}: mean number of jumps {rate:5.2f}; X_1 mean {X.mean():.3f}, variance {X.var():.3f},"
          f" P(X_1 > 1) = {np.mean(X > 1):.3f}, P(X_1 > 3) = {np.mean(X > 3):.4f}")
print(f"exponential law : mean 1, variance 1, P(X > 1) = {math.exp(-1):.3f}, P(X > 3) = {math.exp(-3):.4f}")

eps = 0.1   : mean number of jumps  1.82; X_1 mean 0.991, variance 0.978, P(X_1 > 1) = 0.362, P(X_1 > 3) = 0.0498
eps = 0.001 : mean number of jumps  6.33; X_1 mean 0.997, variance 1.001, P(X_1 > 1) = 0.367, P(X_1 > 3) = 0.0491


eps = 1e-06 : mean number of jumps 13.24; X_1 mean 1.015, variance 1.019, P(X_1 > 1) = 0.377, P(X_1 > 3) = 0.0515
exponential law : mean 1, variance 1, P(X > 1) = 0.368, P(X > 3) = 0.0498


The mean number of jumps grows only like $\log\frac1\varepsilon$ (Exercise 7.3): $1.8$, $6.3$ and $13.2$ jumps for the three thresholds. Replacing the small jumps by their mean loses only their variance $\int_0^\varepsilon x^2\Pi(\mathrm dx)=\int_0^\varepsilon xe^{-x}\,\mathrm dx\approx\frac{\varepsilon^2}2$, which is $0.005$ for $\varepsilon=10^{-1}$, below the Monte Carlo noise: already with two jumps on average, the law is indistinguishable from the exponential law at this sample size.

### P7.6

**How to find it.** Compare $\mathbb E\cos(\xi X)$ with $e^{-|\xi|^\alpha}$, and the quantiles of $(X_1+X_2)/2^{1/\alpha}$ with those of $X_1$. For the Lévy–Itô route with $\alpha=1.5$, the jumps larger than $\varepsilon$ on $[0,1]$ are Poisson with mean $2\varepsilon^{-\alpha}/(\alpha k_\alpha)$, of size $\pm\varepsilon U^{-1/\alpha}$, and the small jumps are replaced by a Gaussian of variance $\int_{|x|\le\varepsilon}x^2\Pi=\frac{2\varepsilon^{2-\alpha}}{(2-\alpha)k_\alpha}$ (the Asmussen–Rosiński approximation). For $\alpha>1$ the compensated large jumps need their mean subtracted, which is $0$ here by symmetry.

In [7]:
import time

def cms(alpha, n):
    V = rng.uniform(-math.pi / 2, math.pi / 2, n); W = rng.exponential(1.0, n)
    if alpha == 1:
        return np.tan(V)
    return np.sin(alpha * V) / np.cos(V)**(1 / alpha) * (np.cos((1 - alpha) * V) / W)**((1 - alpha) / alpha)

for alpha in (0.5, 1.0, 1.5, 1.9):
    X = cms(alpha, 10**6)
    emp = [np.mean(np.cos(xi * X)) for xi in (0.5, 1, 2)]
    print(f"alpha = {alpha}: E cos(xi X) at xi = 0.5, 1, 2: " + ", ".join(f"{v:.4f}" for v in emp)
          + "   exact " + ", ".join(f"{math.exp(-xi**alpha):.4f}" for xi in (0.5, 1, 2)))

alpha = 1.5
X1, X2 = cms(alpha, 10**6), cms(alpha, 10**6)
S = (X1 + X2) / 2**(1 / alpha)
print("quantiles 75%, 95%, 99% of X1:", [round(float(np.quantile(X1, q)), 3) for q in (0.75, 0.95, 0.99)],
      " of (X1+X2)/2^(1/alpha):", [round(float(np.quantile(S, q)), 3) for q in (0.75, 0.95, 0.99)])

k = 2 * math.gamma(1 - alpha) * math.cos(math.pi * alpha / 2) / alpha
def levy_ito(eps, m):
    K = rng.poisson(2 * eps**(-alpha) / (alpha * k), m)
    big = np.array([np.sum(rng.choice([-1, 1], j) * eps * rng.random(j)**(-1 / alpha)) for j in K])
    return big + rng.standard_normal(m) * math.sqrt(2 * eps**(2 - alpha) / ((2 - alpha) * k))

for eps in (0.1, 0.01):
    t0 = time.perf_counter(); Y = levy_ito(eps, 100_000); t1 = time.perf_counter()
    print(f"Levy-Ito, eps = {eps}: E cos(Y) = {np.mean(np.cos(Y)):.4f} (exact {math.exp(-1):.4f}),"
          f" {2 * eps**(-alpha) / (alpha * k):.0f} jumps per sample, {1e6 * (t1 - t0) / 1e5:.1f} microseconds per sample")
t0 = time.perf_counter(); cms(alpha, 100_000); t1 = time.perf_counter()
print(f"Chambers-Mallows-Stuck: {1e6 * (t1 - t0) / 1e5:.2f} microseconds per sample")

alpha = 0.5: E cos(xi X) at xi = 0.5, 1, 2: 0.4932, 0.3682, 0.2430   exact 0.4931, 0.3679, 0.2431
alpha = 1.0: E cos(xi X) at xi = 0.5, 1, 2: 0.6065, 0.3673, 0.1347   exact 0.6065, 0.3679, 0.1353
alpha = 1.5: E cos(xi X) at xi = 0.5, 1, 2: 0.7023, 0.3682, 0.0600   exact 0.7022, 0.3679, 0.0591


alpha = 1.9: E cos(xi X) at xi = 0.5, 1, 2: 0.7644, 0.3674, 0.0233   exact 0.7650, 0.3679, 0.0239
quantiles 75%, 95%, 99% of X1: [0.966, 3.046, 7.672]  of (X1+X2)/2^(1/alpha): [0.966, 3.052, 7.672]


Levy-Ito, eps = 0.1: E cos(Y) = 0.3650 (exact 0.3679), 13 jumps per sample, 13.7 microseconds per sample


Levy-Ito, eps = 0.01: E cos(Y) = 0.3674 (exact 0.3679), 399 jumps per sample, 18.2 microseconds per sample
Chambers-Mallows-Stuck: 0.05 microseconds per sample


The characteristic functions agree to within about $0.001$ for every $\alpha$, and the quantiles of $(X_1+X_2)/2^{1/\alpha}$ match those of $X_1$: the defining property of a stable law. The Lévy–Itô route works too, with a cost proportional to the number of large jumps, $\varepsilon^{-\alpha}$, while Chambers–Mallows–Stuck costs a few arithmetic operations per sample whatever the accuracy: it is exact. The algorithm is J. M. Chambers, C. L. Mallows and B. W. Stuck, *A method for simulating stable random variables*, Journal of the American Statistical Association 71 (1976), 340–344; the Gaussian replacement of the small jumps is justified by S. Asmussen and J. Rosiński, *Approximations of small jumps of Lévy processes with a view towards simulation*, Journal of Applied Probability 38 (2001), 482–493. Stable samplers are used to model heavy-tailed network traffic and financial returns, and in streaming algorithms: Indyk's sketch (2006) estimates the $\ell_p$ norm of a stream from inner products with $p$-stable vectors, because $\sum a_iX_i$ has the law of $\|a\|_pX_1$.